In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool, cv
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
import seaborn as sns
import matplotlib.pyplot as plt

## Grid Search

In [ ]:
# train_size = 100
# train_data = pd.read_csv("/kaggle/input/dmt-2025-2nd-assignment/training_set_VU_DM.csv", nrows=train_size)
train_data = pd.read_csv("/kaggle/input/dmt-2025-2nd-assignment/training_set_VU_DM.csv")

conditions = [
    (train_data['booking_bool'] == 1),                     
    (train_data['click_bool'] == 1) & (train_data['booking_bool'] == 0)  
]
choices = [10, 5]
train_data['score'] = np.select(conditions, choices, default=1)

features = [
    'srch_id',
    'site_id',
    'visitor_location_country_id',
    'prop_country_id',
    'prop_id',
    'prop_starrating',
    'prop_review_score',
    'prop_brand_bool',
    'prop_location_score1',
    'prop_location_score2',
    'price_usd',
    'promotion_flag',
    'srch_destination_id',
    'srch_length_of_stay',
    'srch_booking_window',
    'srch_adults_count',
    'srch_children_count',
    'srch_room_count',
    'srch_saturday_night_bool'
]

# features = [
#     'srch_id',
#     'site_id',
#     'visitor_location_country_id',
#     'prop_country_id',
#     'prop_id',
#     'prop_starrating',
#     'prop_review_score',
#     'prop_brand_bool',
#     'prop_location_score1',
#     'prop_location_score2',
#     'price_usd',
#     'promotion_flag',
#     'srch_destination_id',
#     'srch_length_of_stay',
#     'srch_booking_window',
#     'srch_adults_count',
#     'srch_children_count',
#     'srch_room_count',
#     'srch_saturday_night_bool',
#     'comp1_rate', 'comp1_inv', 'comp1_rate_percent_diff',
#     'comp2_rate', 'comp2_inv', 'comp2_rate_percent_diff',
#     'comp3_rate', 'comp3_inv', 'comp3_rate_percent_diff',
#     'comp4_rate', 'comp4_inv', 'comp4_rate_percent_diff',
#     'comp5_rate', 'comp5_inv', 'comp5_rate_percent_diff',
#     'comp6_rate', 'comp6_inv', 'comp6_rate_percent_diff',
#     'comp7_rate', 'comp7_inv', 'comp7_rate_percent_diff',
#     'comp8_rate', 'comp8_inv', 'comp8_rate_percent_diff'
# ]

# comp_features = [
#     'comp1_rate', 'comp1_inv', 'comp1_rate_percent_diff',
#     'comp2_rate', 'comp2_inv', 'comp2_rate_percent_diff',
#     'comp3_rate', 'comp3_inv', 'comp3_rate_percent_diff',
#     'comp4_rate', 'comp4_inv', 'comp4_rate_percent_diff',
#     'comp5_rate', 'comp5_inv', 'comp5_rate_percent_diff',
#     'comp6_rate', 'comp6_inv', 'comp6_rate_percent_diff',
#     'comp7_rate', 'comp7_inv', 'comp7_rate_percent_diff',
#     'comp8_rate', 'comp8_inv', 'comp8_rate_percent_diff'
# ]
# train_data[comp_features] = train_data[comp_features].fillna(0)

categorical_features = [
    'site_id',
    'visitor_location_country_id',
    'prop_country_id',
    'prop_id',  
    'srch_destination_id',
    'promotion_flag',
    'srch_saturday_night_bool'
]

X = train_data[features]
y = train_data['score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

sample_weights = np.where(y == 10, 10, np.where(y == 5, 5, 1))
train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=categorical_features,
    weight=sample_weights[y_train.index]
)

learning_rates = [0.01, 0.05, 0.1]
depths = [8, 10, 12, 14]
l2_leaf_regs = [1,3,5]

print('Start grid search')
for lr in learning_rates:
    for depth in depths:
        for l2 in l2_leaf_regs:
            model = CatBoostRegressor(
                iterations=500,
                task_type="GPU",
                devices='0,1',
                eval_metric='RMSE',
                learning_rate=lr,
                depth=depth,
                l2_leaf_reg=l2,
                cat_features=categorical_features,
                verbose=0
            )
            
            model.fit(
                X_train, 
                y_train,
                sample_weight=sample_weights[y_train.index],
                verbose=0
            )
            
            y_pred = model.predict(X_test)
            rmse = (mean_squared_error(y_test, y_pred))**0.5
            
            print("Parameters:")
            print(f"  learning_rate: {lr}")
            print(f"  l2_leaf_reg:   {l2}")
            print(f"  depth:         {depth}")
            print(f"Test RMSE:       {rmse:.6f}")
            print("-" * 40)
            print("-" * 40)

In [ ]:
data = [
    {'learning_rate': 0.01, 'l2_leaf_reg': 1, 'depth': 8, 'Test_RMSE': 2.262969},
    {'learning_rate': 0.01, 'l2_leaf_reg': 3, 'depth': 8, 'Test_RMSE': 2.261451},
    {'learning_rate': 0.01, 'l2_leaf_reg': 5, 'depth': 8, 'Test_RMSE': 2.261516},
    {'learning_rate': 0.01, 'l2_leaf_reg': 1, 'depth': 10, 'Test_RMSE': 2.263714},
    {'learning_rate': 0.01, 'l2_leaf_reg': 3, 'depth': 10, 'Test_RMSE': 2.269255},
    {'learning_rate': 0.01, 'l2_leaf_reg': 5, 'depth': 10, 'Test_RMSE': 2.270707},
    {'learning_rate': 0.01, 'l2_leaf_reg': 1, 'depth': 12, 'Test_RMSE': 2.260561},
    {'learning_rate': 0.01, 'l2_leaf_reg': 3, 'depth': 12, 'Test_RMSE': 2.261733},
    {'learning_rate': 0.01, 'l2_leaf_reg': 5, 'depth': 12, 'Test_RMSE': 2.262665},
    {'learning_rate': 0.01, 'l2_leaf_reg': 1, 'depth': 14, 'Test_RMSE': 2.239716},
    {'learning_rate': 0.01, 'l2_leaf_reg': 3, 'depth': 14, 'Test_RMSE': 2.240564},
    {'learning_rate': 0.01, 'l2_leaf_reg': 5, 'depth': 14, 'Test_RMSE': 2.243175},
    {'learning_rate': 0.05, 'l2_leaf_reg': 1, 'depth': 8, 'Test_RMSE': 2.220080},
    {'learning_rate': 0.05, 'l2_leaf_reg': 3, 'depth': 8, 'Test_RMSE': 2.217398},
    {'learning_rate': 0.05, 'l2_leaf_reg': 5, 'depth': 8, 'Test_RMSE': 2.216496},
    {'learning_rate': 0.05, 'l2_leaf_reg': 1, 'depth': 10, 'Test_RMSE': 2.209896},
    {'learning_rate': 0.05, 'l2_leaf_reg': 3, 'depth': 10, 'Test_RMSE': 2.202995},
    {'learning_rate': 0.05, 'l2_leaf_reg': 5, 'depth': 10, 'Test_RMSE': 2.208908},
    {'learning_rate': 0.05, 'l2_leaf_reg': 1, 'depth': 12, 'Test_RMSE': 2.159989},
    {'learning_rate': 0.05, 'l2_leaf_reg': 3, 'depth': 12, 'Test_RMSE': 2.166873},
    {'learning_rate': 0.05, 'l2_leaf_reg': 5, 'depth': 12, 'Test_RMSE': 2.167720},
    {'learning_rate': 0.05, 'l2_leaf_reg': 1, 'depth': 14, 'Test_RMSE': 2.061946},
    {'learning_rate': 0.05, 'l2_leaf_reg': 3, 'depth': 14, 'Test_RMSE': 2.072173},
    {'learning_rate': 0.05, 'l2_leaf_reg': 5, 'depth': 14, 'Test_RMSE': 2.080781},
    {'learning_rate': 0.1, 'l2_leaf_reg': 1, 'depth': 8, 'Test_RMSE': 2.200120},
    {'learning_rate': 0.1, 'l2_leaf_reg': 3, 'depth': 8, 'Test_RMSE': 2.197164},
    {'learning_rate': 0.1, 'l2_leaf_reg': 5, 'depth': 8, 'Test_RMSE': 2.196261},
    {'learning_rate': 0.1, 'l2_leaf_reg': 1, 'depth': 10, 'Test_RMSE': 2.176342},
    {'learning_rate': 0.1, 'l2_leaf_reg': 3, 'depth': 10, 'Test_RMSE': 2.174415},
    {'learning_rate': 0.1, 'l2_leaf_reg': 5, 'depth': 10, 'Test_RMSE': 2.173154},
    {'learning_rate': 0.1, 'l2_leaf_reg': 1, 'depth': 12, 'Test_RMSE': 2.111213},
    {'learning_rate': 0.1, 'l2_leaf_reg': 3, 'depth': 12, 'Test_RMSE': 2.113071},
    {'learning_rate': 0.1, 'l2_leaf_reg': 5, 'depth': 12, 'Test_RMSE': 2.116139},
    {'learning_rate': 0.1, 'l2_leaf_reg': 1, 'depth': 14, 'Test_RMSE': 1.982188},
    {'learning_rate': 0.1, 'l2_leaf_reg': 3, 'depth': 14, 'Test_RMSE': 1.986139},
    {'learning_rate': 0.1, 'l2_leaf_reg': 5, 'depth': 14, 'Test_RMSE': 1.992885},
]

df = pd.DataFrame(data)

fig = plt.figure(figsize=(10, 10), dpi=300) 

g = sns.FacetGrid(df, col="depth", height=4, aspect=1, col_wrap=2)
def heatmap(data, **kwargs):
    pivot = data.pivot(index='l2_leaf_reg', columns='learning_rate', values='Test_RMSE')
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu", cbar=True, **kwargs)

g.map_dataframe(heatmap)
g.set_axis_labels("Learning Rate", "L2 Leaf Reg", fontsize=14)
g.set_titles("Depth = {col_name}", fontsize=20)
plt.subplots_adjust(top=0.9)
g.fig.suptitle("RMSE for Catboost with different parameters", fontsize=16)

plt.savefig("catboost_grid_search_2x2.png")
plt.show()


## With no competition varibales


In [ ]:
train_data = pd.read_csv("/kaggle/input/dmt-2025-2nd-assignment/training_set_VU_DM.csv")
test_data = pd.read_csv("/kaggle/input/dmt-2025-2nd-assignment/test_set_VU_DM.csv")

conditions = [
    (train_data['booking_bool'] == 1),                     
    (train_data['click_bool'] == 1) & (train_data['booking_bool'] == 0)  
]
choices = [10, 5]
train_data['score'] = np.select(conditions, choices, default=1)

features = [
    'srch_id',
    'site_id',
    'visitor_location_country_id',
    'prop_country_id',
    'prop_id',
    'prop_starrating',
    'prop_review_score',
    'prop_brand_bool',
    'prop_location_score1',
    'prop_location_score2',
    'price_usd',
    'promotion_flag',
    'srch_destination_id',
    'srch_length_of_stay',
    'srch_booking_window',
    'srch_adults_count',
    'srch_children_count',
    'srch_room_count',
    'srch_saturday_night_bool'
]

categorical_features = [
    'site_id',
    'visitor_location_country_id',
    'prop_country_id',
    'prop_id',  
    'srch_destination_id',
    'promotion_flag',
    'srch_saturday_night_bool'
]

X = train_data[features]
y = train_data['score']
sample_weights = np.where(y == 10, 10, np.where(y == 5, 5, 1))

def catboost_simulation(iter_num, lr, depth, l2, X, y, X_test, test_data, sample_weights, categorical_features):
    model = CatBoostRegressor(
        iterations=iter_num,
        task_type="GPU",
        devices='0,1',
        learning_rate=lr,
        depth=depth,
        l2_leaf_reg=l2,
        eval_metric='RMSE',
        verbose=100
    )
    
    model.fit(
        X, y,
        sample_weight=sample_weights,
        cat_features=categorical_features,
        verbose=100
    )
    
    test_predictions = model.predict(X_test)
    result = test_data[['srch_id', 'prop_id']].copy()
    result['predicted_score'] = test_predictions
    sorted_result = result.sort_values(
        by=['srch_id', 'predicted_score'],
        ascending=[True, False]
    ).reset_index(drop=True)
    sorted_result[['srch_id', 'prop_id']].to_csv(
        f"catboost_predictions_iter_{iter_num}lr_{lr}_depth_{depth}_l2_{l2}.csv", index=False)
    print('Saved successfully!')

In [ ]:
for depth in [8,10]: 
    for lr in [0.01,0.05]: 
        catboost_simulation(
            iter_num=1000,
            lr=lr,
            depth=depth,
            l2=3,
            X=X,
            y=y,
            X_test=test_data[features],
            test_data=test_data,
            sample_weights=sample_weights,
            categorical_features=categorical_features
        )

## With competition variables

In [ ]:
# # train_size = 100
# # train_data = pd.read_csv("/kaggle/input/dmt-2025-2nd-assignment/training_set_VU_DM.csv", nrows=train_size)
# train_data = pd.read_csv("/kaggle/input/dmt-2025-2nd-assignment/training_set_VU_DM.csv")

# conditions = [
#     (train_data['booking_bool'] == 1),                     
#     (train_data['click_bool'] == 1) & (train_data['booking_bool'] == 0)  
# ]
# choices = [10, 5]
# train_data['score'] = np.select(conditions, choices, default=0)

# # features = [
# #     'srch_id',
# #     'site_id',
# #     'visitor_location_country_id',
# #     'prop_country_id',
# #     'prop_id',
# #     'prop_starrating',
# #     'prop_review_score',
# #     'prop_brand_bool',
# #     'prop_location_score1',
# #     'prop_location_score2',
# #     'price_usd',
# #     'promotion_flag',
# #     'srch_destination_id',
# #     'srch_length_of_stay',
# #     'srch_booking_window',
# #     'srch_adults_count',
# #     'srch_children_count',
# #     'srch_room_count',
# #     'srch_saturday_night_bool'
# # ]

# features = [
#     'srch_id',
#     'site_id',
#     'visitor_location_country_id',
#     'prop_country_id',
#     'prop_id',
#     'prop_starrating',
#     'prop_review_score',
#     'prop_brand_bool',
#     'prop_location_score1',
#     'prop_location_score2',
#     'price_usd',
#     'promotion_flag',
#     'srch_destination_id',
#     'srch_length_of_stay',
#     'srch_booking_window',
#     'srch_adults_count',
#     'srch_children_count',
#     'srch_room_count',
#     'srch_saturday_night_bool',
#     'comp1_rate', 'comp1_inv', 'comp1_rate_percent_diff',
#     'comp2_rate', 'comp2_inv', 'comp2_rate_percent_diff',
#     'comp3_rate', 'comp3_inv', 'comp3_rate_percent_diff',
#     'comp4_rate', 'comp4_inv', 'comp4_rate_percent_diff',
#     'comp5_rate', 'comp5_inv', 'comp5_rate_percent_diff',
#     'comp6_rate', 'comp6_inv', 'comp6_rate_percent_diff',
#     'comp7_rate', 'comp7_inv', 'comp7_rate_percent_diff',
#     'comp8_rate', 'comp8_inv', 'comp8_rate_percent_diff'
# ]

# comp_features = [
#     'comp1_rate', 'comp1_inv', 'comp1_rate_percent_diff',
#     'comp2_rate', 'comp2_inv', 'comp2_rate_percent_diff',
#     'comp3_rate', 'comp3_inv', 'comp3_rate_percent_diff',
#     'comp4_rate', 'comp4_inv', 'comp4_rate_percent_diff',
#     'comp5_rate', 'comp5_inv', 'comp5_rate_percent_diff',
#     'comp6_rate', 'comp6_inv', 'comp6_rate_percent_diff',
#     'comp7_rate', 'comp7_inv', 'comp7_rate_percent_diff',
#     'comp8_rate', 'comp8_inv', 'comp8_rate_percent_diff'
# ]
# train_data[comp_features] = train_data[comp_features].fillna(0)

# categorical_features = [
#     'site_id',
#     'visitor_location_country_id',
#     'prop_country_id',
#     'prop_id',  
#     'srch_destination_id',
#     'promotion_flag',
#     'srch_saturday_night_bool'
# ]

# X = train_data[features]
# y = train_data['score']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# sample_weights = np.where(y == 10, 10, np.where(y == 5, 5, 1))
# train_pool = Pool(
#     data=X_train,
#     label=y_train,
#     cat_features=categorical_features,
#     weight=sample_weights[y_train.index]
# )

# learning_rates = [0.1]
# depths = [20]
# l2_leaf_regs = [5]

# print('Start grid search')
# for lr in learning_rates:
#     for depth in depths:
#         for l2 in l2_leaf_regs:
#             model = CatBoostRegressor(
#                 iterations=1000,
#                 task_type="GPU",
#                 devices='0,1',
#                 eval_metric='RMSE',
#                 learning_rate=lr,
#                 depth=depth,
#                 l2_leaf_reg=l2,
#                 cat_features=categorical_features,
#                 verbose=0
#             )
            
#             model.fit(
#                 X_train, 
#                 y_train,
#                 sample_weight=sample_weights[y_train.index],
#                 verbose=0
#             )
            
#             y_pred = model.predict(X_test)
#             rmse = (mean_squared_error(y_test, y_pred))**0.5
            
#             print("Parameters:")
#             print(f"  learning_rate: {lr}")
#             print(f"  l2_leaf_reg:   {l2}")
#             print(f"  depth:         {depth}")
#             print(f"Test RMSE:       {rmse:.6f}")
#             print("-" * 40)
#             print("-" * 40)